# 02 · Feature-relation audit and detectors (NF-UNSW-NB15-v3, NF-ToN-IoT-v3, NF-CSE-CIC-IDS2018-v3)

Design decisions:
- Inputs: the notebook 01 parquet files (`XIDS_Certify_Research/data/nf3_v1/`). Only the detector_train and development roles are read here; calibration and locked_test stay untouched.
- Relation audit: every candidate relation between NetFlow fields (durations, packet counts, byte bounds, the packet-size histogram, throughput, per-second byte fields, TCP flags, TTL order, inter-arrival statistics, retransmissions) is checked on detector_train rows, both as exact formulas and as brackets. A bracket allows for fields computed from the exact duration while the file stores whole milliseconds. The attacker operation model in notebook 03 may use a relation only if it holds on at least 99.9% of applicable rows in both datasets. Field names alone are not trusted.
- Detector inputs: the 46 data-stage features minus MIN_TTL and MAX_TTL (44 features). TTL is set by the sending host and drops by one per router hop, so in a testbed it identifies the sending machine, and an attacker sets it with one socket option. The first run of this notebook found each TTL field separates NF-UNSW-NB15-v3 classes with AUROC 0.9994 on its own and carries 99.4% of that detector's gain; they are excluded on both datasets, like IP addresses and ports.
- Detector: one XGBoost model per dataset (300 trees at most, depth 6, early stopping on development), trained on distinct records without weights. The saved model is trimmed to its best round, so the verified model and the deciding model are the same object.
- Operating threshold: the score giving a 1% false-alarm rate on development benign flows, fixed here and reused unchanged later.
- Metrics on development, reported per distinct record and weighted by multiplicity (how often each record occurred in the raw traffic).
- Shortcut check: how well each feature separates the classes on its own; a feature near 1.0 is reported as a shortcut risk.
- Models are saved outside the repository in `XIDS_Certify_Research/models/nf02/`; their SHA-256 and the thresholds are committed.
- NF-CSE-CIC-IDS2018-v3 was added after the first two datasets and before any certificate result. The relation verdict covers all three datasets, so the attacker model rests only on relations that hold everywhere the certificate will run.
- A dataset whose audit rows, model file and model record already exist is not redone and its committed results are kept, unless the detector configuration has changed (then everything is redone) or `FORCE = True`. Kept model files are checked against their recorded SHA-256.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import gc, hashlib, importlib, json, os, sys
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/XIDS_Certify_Research')
REPO = PROJECT / 'xids-certify'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import sklearn
import xgboost
import src.nf3_data as nf
import src.relations as rel
import src.detector as det
for m in (nf, rel, det):
    importlib.reload(m)

TABLES = REPO / 'results' / 'tables'
MODELS = PROJECT / 'models' / 'nf02'
DATASETS = ['unsw', 'ton', 'cic2018']
FORCE = False                                # True redoes the audit and the detector for every dataset
FEATS = nf.MODEL_FEATURES
DET_FEATS = det.detector_features(FEATS)
print(f'detector inputs: {len(DET_FEATS)} features (excluded: {sorted(set(FEATS) - set(DET_FEATS))})')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 70)
VERSIONS = {'pandas': pd.__version__, 'numpy': np.__version__, 'xgboost': xgboost.__version__, 'sklearn': sklearn.__version__}
print(VERSIONS)
summary01 = json.loads((TABLES / 'nf01_summary.json').read_text())
print('data split hashes:', summary01['split_hash'])

det_config_hash = det.write_config(REPO)
prev_models = json.loads((TABLES / 'nf02_models.json').read_text()) if (TABLES / 'nf02_models.json').exists() else {}
prev_rel = pd.read_csv(TABLES / 'nf02_relations.csv') if (TABLES / 'nf02_relations.csv').exists() else pd.DataFrame(columns=['dataset'])
if prev_models and prev_models.get('detector_config_sha256') != det_config_hash:
    print('detector configuration changed since the last run: every dataset will be redone')
    FORCE = True
for d in DATASETS:
    if d in prev_models and (MODELS / f'{d}_xgb.json').exists() and not FORCE:
        assert hashlib.sha256((MODELS / f'{d}_xgb.json').read_bytes()).hexdigest() == prev_models[d]['model_sha256'],             f'{d}: model file on Drive does not match its recorded SHA-256'
TODO_REL = [d for d in DATASETS if FORCE or d not in set(prev_rel['dataset'])]
TODO_DET = [d for d in DATASETS if FORCE or d not in prev_models or not (MODELS / f'{d}_xgb.json').exists()]
print(f'relation audit to run: {TODO_REL} | detectors to train: {TODO_DET}')

## Feature-relation audit (detector_train rows only)

In [ ]:
audits = [prev_rel[prev_rel['dataset'].isin([d for d in DATASETS if d not in TODO_REL])]]
for ds in TODO_REL:
    df = nf.load_split(PROJECT, ds, roles=['detector_train'], columns=FEATS + ['flow_start_ms', 'flow_end_ms'])
    audits.append(rel.audit(df, ds))
    print(f'{ds}: {len(df):,} rows audited')
    del df
    gc.collect()
relations = pd.concat(audits, ignore_index=True)
relations = relations[relations['dataset'].isin(DATASETS)]
verdict = rel.verdict(relations)
display(verdict.sort_values('usable_in_operation_model', ascending=False))

In [ ]:
# ratio relations: the quantiles show the true scale factor when the named formula is wrong
display(relations[relations['kind'] == 'ratio = 1'][['dataset', 'relation', 'rows_applicable', 'share_satisfied', 'q01', 'q50', 'q99']])

## Detectors: train on detector_train, early stopping and threshold on development

In [ ]:
def run_detector(ds):
    cols = DET_FEATS + ['y', 'family', 'multiplicity', 'role']
    df = nf.load_split(PROJECT, ds, roles=['detector_train', 'development'], columns=cols)
    tr, dv = df[df['role'] == 'detector_train'], df[df['role'] == 'development']
    booster = det.fit(tr[DET_FEATS], tr['y'].to_numpy(), dv[DET_FEATS], dv['y'].to_numpy())
    s_dev = det.predict(booster, dv[DET_FEATS])
    y_dev = dv['y'].to_numpy()
    thr = det.threshold_at_fpr(s_dev[y_dev == 0])
    m_rec, fam_rec = det.metrics(y_dev, s_dev, thr, family=dv['family'].to_numpy())
    m_w, fam_w = det.metrics(y_dev, s_dev, thr, weight=dv['multiplicity'].to_numpy(), family=dv['family'].to_numpy())
    shortcut = det.single_feature_auroc(tr[DET_FEATS], tr['y'].to_numpy()).assign(dataset=ds)
    imp = det.importance(booster, DET_FEATS).assign(dataset=ds)
    path = MODELS / f'{ds}_xgb.json'
    sha = det.save(booster, path)
    stats = det.tree_stats(booster)
    rec = {'dataset': ds, 'model_path': str(path.relative_to(PROJECT)), 'model_sha256': sha, 'threshold': thr,
           'features': DET_FEATS,
           'train_rows': int(len(tr)), 'development_rows': int(len(dv)), **stats, 'versions': VERSIONS}
    metrics = pd.DataFrame([{'dataset': ds, 'weighting': 'distinct records', **m_rec},
                            {'dataset': ds, 'weighting': 'multiplicity', **m_w}])
    fams = pd.concat([fam_rec.assign(dataset=ds, weighting='distinct records'),
                      fam_w.assign(dataset=ds, weighting='multiplicity')], ignore_index=True)
    del df, tr, dv
    gc.collect()
    return rec, metrics, fams, shortcut, imp

results = {}

In [ ]:
def run_or_keep(ds):
    if ds not in TODO_DET:
        print(f"{ds}: committed model kept (SHA-256 {prev_models[ds]['model_sha256'][:16]}..., threshold {prev_models[ds]['threshold']:.6g})")
        return
    results[ds] = run_detector(ds)
    rec, metrics, fams, shortcut, imp = results[ds]
    print(json.dumps({k: v for k, v in rec.items() if k not in ('versions', 'features')}, indent=2))
    display(metrics)
    display(fams)

In [ ]:
run_or_keep('unsw')

In [ ]:
run_or_keep('ton')

In [ ]:
run_or_keep('cic2018')

## Shortcut check: features that separate the classes on their own

In [ ]:
def merged(name, part):
    path = TABLES / name
    old = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=['dataset'])
    old = old[old['dataset'].isin([d for d in DATASETS if d not in results])]
    fresh = [results[d][part] for d in DATASETS if d in results]
    out = pd.concat([old] + fresh, ignore_index=True) if fresh else old
    return out.sort_values('dataset', key=lambda c: c.map({d: i for i, d in enumerate(DATASETS)}), kind='stable')

shortcuts = merged('nf02_single_feature_auroc.csv', 3)
importances = merged('nf02_importance.csv', 4)
for ds in DATASETS:
    s = shortcuts[shortcuts['dataset'] == ds]
    print(f"{ds}: features with AUROC alone >= 0.99: {s.loc[s['auroc_alone'] >= 0.99, 'feature'].tolist()}")
    display(s.head(10))
    display(importances[importances['dataset'] == ds].head(10))

## Tables, model hashes and configuration

In [ ]:
relations.to_csv(TABLES / 'nf02_relations.csv', index=False)
verdict.to_csv(TABLES / 'nf02_relations_verdict.csv', index=False)
merged('nf02_detector_metrics.csv', 1).to_csv(TABLES / 'nf02_detector_metrics.csv', index=False)
merged('nf02_family_flag_rates.csv', 2).to_csv(TABLES / 'nf02_family_flag_rates.csv', index=False)
shortcuts.to_csv(TABLES / 'nf02_single_feature_auroc.csv', index=False)
importances.to_csv(TABLES / 'nf02_importance.csv', index=False)
models = {d: (results[d][0] if d in results else prev_models[d]) for d in DATASETS}
models['detector_config_sha256'] = det_config_hash
models['data_split_hash'] = summary01['split_hash']
models['usable_relations'] = sorted(verdict.loc[verdict['usable_in_operation_model'], 'relation'].tolist())
(TABLES / 'nf02_models.json').write_text(json.dumps(models, indent=2) + '\n')
display(pd.read_csv(TABLES / 'nf02_detector_metrics.csv'))
print('usable relations:', models['usable_relations'])

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/XIDS_Certify_Research/xids-certify")
msg = "Notebook 02: NF-CSE-CIC-IDS2018-v3 added (relation audit and detector) before any certificate result; relation verdict now covers all three datasets; NF-UNSW-NB15-v3 and NF-ToN-IoT-v3 models and tables kept unchanged and checked against their recorded SHA-256"
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)